## Advanced Machine Learning & Deep Learning Modeling with Performance Optimization


In [21]:
%%capture
# !pip -q install -U transformers scikit-learn

In [22]:
import csv
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    pipeline,
)

pd.set_option("display.max_colwidth", 240)

In [23]:
SEED = 42
MODEL_NAME = "bert-base-uncased"
PIPELINE_MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
VAL_SIZE = 0.10
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
PIPELINE_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

RAW_LABEL_TO_ID = {1: 0, 2: 1}
ID_TO_LABEL_NAME = {0: "negative", 1: "positive"}
LABEL_NAME_TO_ID = {"negative": 0, "positive": 1}

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type != "cuda":
    print("GPU is not enabled. The notebook will still run, but training will be much slower.")


Using device: cuda


In [24]:
required_files = ["train_0.01.csv", "test_0.01.csv", "readme.txt"]
missing_files = [fname for fname in required_files if not Path(fname).exists()]

if missing_files:
    print("Missing files:", ", ".join(missing_files))
    try:
        from google.colab import files
        print("Please upload the missing files now.")
        _ = files.upload()
    except ImportError as e:
        raise FileNotFoundError(
            f"Could not find {missing_files}. Put the files in the current working directory and rerun this cell."
        ) from e

missing_files = [fname for fname in required_files if not Path(fname).exists()]
if missing_files:
    raise FileNotFoundError(f"Still missing required files: {missing_files}")

train_path = Path("train_0.01.csv")
test_path = Path("test_0.01.csv")
readme_path = Path("readme.txt")
print("Found files:")
print(" -", train_path.resolve())
print(" -", test_path.resolve())
print(" -", readme_path.resolve())


Found files:
 - /content/train_0.01.csv
 - /content/test_0.01.csv
 - /content/readme.txt


In [25]:
readme_text = readme_path.read_text(encoding="utf-8", errors="ignore")

print("Loaded readme.txt")
print("-" * 80)
print(readme_text)

readme_summary_df = pd.DataFrame(
    {
        "item": [
            "csv_header",
            "raw_label_1",
            "raw_label_2",
            "column_1",
            "column_2",
        ],
        "value": [
            "no header row",
            "negative",
            "positive",
            "integer class label",
            "full review text",
        ],
    }
)

display(readme_summary_df)


Loaded readme.txt
--------------------------------------------------------------------------------
Yelp Review Polarity Dataset

Version 1, Updated 09/09/2015

ORIGIN

The Yelp reviews dataset consists of reviews from Yelp. It is extracted from the Yelp Dataset Challenge 2015 data. For more information, please refer to http://www.yelp.com/dataset_challenge

The Yelp reviews polarity dataset is constructed by Xiang Zhang (xiang.zhang@nyu.edu) from the above dataset. It is first used as a text classification benchmark in the following paper: Xiang Zhang, Junbo Zhao, Yann LeCun. Character-level Convolutional Networks for Text Classification. Advances in Neural Information Processing Systems 28 (NIPS 2015).


DESCRIPTION

The Yelp reviews polarity dataset is constructed by considering stars 1 and 2 negative, and 3 and 4 positive. For each polarity 280,000 training samples and 19,000 testing samples are take randomly. In total there are 560,000 trainig samples and 38,000 testing samples. Ne

,item,value
0,csv_header,no header row
1,raw_label_1,negative
2,raw_label_2,positive
3,column_1,integer class label
4,column_2,full review text


In [26]:
train_df = pd.read_csv(
    train_path,
    header=None,
    names=["raw_label", "text"],
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
)

test_df = pd.read_csv(
    test_path,
    header=None,
    names=["raw_label", "text"],
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
)

for df_name, df in [("train", train_df), ("test", test_df)]:
    if df["raw_label"].isna().any() or df["text"].isna().any():
        raise ValueError(f"{df_name} contains missing labels or text entries.")
    df["label"] = df["raw_label"].map(RAW_LABEL_TO_ID)
    if df["label"].isna().any():
        bad = sorted(df.loc[df["label"].isna(), "raw_label"].unique().tolist())
        raise ValueError(f"Unexpected labels found in {df_name}: {bad}")
    df["label"] = df["label"].astype(int)
    df["text"] = df["text"].astype(str)

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
train_df.head()

Train shape: (5600, 3)
Test shape:  (380, 3)


,raw_label,text,label
0,1,"Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually...",0
1,2,"Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff...",1
2,1,"I don't know what Dr. Goldberg was like before moving to Arizona, but let me tell you, STAY AWAY from this doctor and this office. I was going to Dr. Johnson before he left and Goldberg took over when Johnson left. He is not a caring d...",0
3,1,"I'm writing this review to give you a heads up before you see this Doctor. The office staff and administration are very unprofessional. I left a message with multiple people regarding my bill, and no one ever called me back. I had to ho...",0
4,2,"All the food is great here. But the best thing they have is their wings. Their wings are simply fantastic!! The \""Wet Cajun\"" are by the best & most popular. I also like the seasoned salt wings. Wing Night is Monday & Wednesday night...",1


In [27]:
def summarize_split(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "split": [name],
            "rows": [len(df)],
            "negative_count": [(df["label"] == 0).sum()],
            "positive_count": [(df["label"] == 1).sum()],
            "avg_chars": [df["text"].str.len().mean()],
            "median_chars": [df["text"].str.len().median()],
        }
    )

overview_df = pd.concat(
    [
        summarize_split(train_df, "train_full"),
        summarize_split(test_df, "test"),
    ],
    ignore_index=True,
)

display(overview_df)

label_note = pd.DataFrame(
    {
        "raw_label_in_csv": [1, 2],
        "meaning": ["negative", "positive"],
        "model_label_id": [0, 1],
    }
)
display(label_note)

,split,rows,negative_count,positive_count,avg_chars,median_chars
0,train_full,5600,2919,2681,733.154464,549.0
1,test,380,181,199,740.742105,575.0


,raw_label_in_csv,meaning,model_label_id
0,1,negative,0
1,2,positive,1


In [28]:
train_split_df, val_df = train_test_split(
    train_df[["text", "label"]].copy(),
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df["label"],
)

train_split_df = train_split_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_eval_df = test_df[["text", "label"]].copy().reset_index(drop=True)

split_sizes_df = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_split_df), len(val_df), len(test_eval_df)],
        "negative_count": [
            (train_split_df["label"] == 0).sum(),
            (val_df["label"] == 0).sum(),
            (test_eval_df["label"] == 0).sum(),
        ],
        "positive_count": [
            (train_split_df["label"] == 1).sum(),
            (val_df["label"] == 1).sum(),
            (test_eval_df["label"] == 1).sum(),
        ],
    }
)

display(split_sizes_df)

,split,rows,negative_count,positive_count
0,train,5040,2627,2413
1,validation,560,292,268
2,test,380,181,199


In [29]:
hyperparams_df = pd.DataFrame(
    {
        "hyperparameter": [
            "model_name",
            "validation_fraction",
            "max_length",
            "train_batch_size",
            "eval_batch_size",
            "learning_rate",
            "weight_decay",
            "num_epochs",
            "warmup_ratio",
            "gradient_clip_norm",
            "optimizer",
            "early_stopping_patience",
            "random_seed",
        ],
        "value": [
            MODEL_NAME,
            VAL_SIZE,
            MAX_LENGTH,
            TRAIN_BATCH_SIZE,
            EVAL_BATCH_SIZE,
            LEARNING_RATE,
            WEIGHT_DECAY,
            NUM_EPOCHS,
            WARMUP_RATIO,
            GRAD_CLIP_NORM,
            "AdamW",
            EARLY_STOPPING_PATIENCE,
            SEED,
        ],
    }
)

display(hyperparams_df)


,hyperparameter,value
0,model_name,bert-base-uncased
1,validation_fraction,0.1
2,max_length,256
3,train_batch_size,16
4,eval_batch_size,32
5,learning_rate,0.00002
6,weight_decay,0.01
7,num_epochs,3
8,warmup_ratio,0.1
9,gradient_clip_norm,1.0


In [30]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            max_length=max_length,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx] for key in self.encodings}
        item["labels"] = int(self.labels[idx])
        return item

train_dataset = ReviewDataset(
    train_split_df["text"].tolist(),
    train_split_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

val_dataset = ReviewDataset(
    val_df["text"].tolist(),
    val_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

test_dataset = ReviewDataset(
    test_eval_df["text"].tolist(),
    test_eval_df["label"].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

num_workers = 2 if device.type == "cuda" else 0
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 315
Validation batches: 18
Test batches: 12


In [31]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID_TO_LABEL_NAME,
    label2id=LABEL_NAME_TO_ID,
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_training_steps = len(train_loader) * NUM_EPOCHS
num_warmup_steps = int(WARMUP_RATIO * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_training_steps,
)

BEST_STATE_PATH = Path("best_bert_yelp_state.pt")
BEST_EXPORT_DIR = Path("bert_yelp_sentiment_best")

print(f"Total training steps: {total_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total training steps: 945
Warmup steps: 94


In [32]:
def compute_binary_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )
    return {
        "accuracy": float(acc),
        "precision_positive": float(precision),
        "recall_positive": float(recall),
        "f1_positive": float(f1),
    }

@torch.no_grad()
def predict_with_probs(model, dataloader, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_pos_probs = []
    all_confidences = []

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

        all_labels.extend(batch["labels"].cpu().numpy().tolist())
        all_preds.extend(preds.cpu().numpy().tolist())
        all_pos_probs.extend(probs[:, 1].cpu().numpy().tolist())
        all_confidences.extend(probs.max(dim=-1).values.cpu().numpy().tolist())

    return (
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_pos_probs),
        np.array(all_confidences),
    )

def evaluate_model(model, dataloader, device):
    y_true, y_pred, pos_probs, confidences = predict_with_probs(model, dataloader, device)
    metrics = compute_binary_metrics(y_true, y_pred)
    return metrics, y_true, y_pred, pos_probs, confidences

In [34]:
history = []
best_val_f1 = -1.0
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = running_loss / max(len(train_loader), 1)

    val_metrics, _, _, _, _ = evaluate_model(model, val_loader, device)
    epoch_record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(epoch_record)

    print(
        f"Epoch {epoch}: "
        f"train_loss={avg_train_loss:.4f}, "
        f"val_accuracy={val_metrics['accuracy']:.4f}, "
        f"val_precision={val_metrics['precision_positive']:.4f}, "
        f"val_recall={val_metrics['recall_positive']:.4f}, "
        f"val_f1={val_metrics['f1_positive']:.4f}"
    )

    if val_metrics["f1_positive"] > best_val_f1:
        best_val_f1 = val_metrics["f1_positive"]
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_STATE_PATH)
        print(f"  -> Saved new best model to {BEST_STATE_PATH}")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement. Patience counter: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

if not BEST_STATE_PATH.exists():
    raise FileNotFoundError("Best model checkpoint was not saved correctly.")

model.load_state_dict(torch.load(BEST_STATE_PATH, map_location=device))
model.to(device)

history_df = pd.DataFrame(history)
display(history_df)

Epoch 1/3:   0%|          | 0/315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 1: train_loss=0.3207, val_accuracy=0.9411, val_precision=0.9304, val_recall=0.9478, val_f1=0.9390
  -> Saved new best model to best_bert_yelp_state.pt


Epoch 2/3:   0%|          | 0/315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 2: train_loss=0.1443, val_accuracy=0.9482, val_precision=0.9476, val_recall=0.9440, val_f1=0.9458
  -> Saved new best model to best_bert_yelp_state.pt


Epoch 3/3:   0%|          | 0/315 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 3: train_loss=0.0606, val_accuracy=0.9536, val_precision=0.9481, val_recall=0.9552, val_f1=0.9517
  -> Saved new best model to best_bert_yelp_state.pt


,epoch,train_loss,val_accuracy,val_precision_positive,val_recall_positive,val_f1_positive
0,1,0.320672,0.941071,0.930403,0.947761,0.939002
1,2,0.144301,0.948214,0.947566,0.944030,0.945794
2,3,0.060566,0.953571,0.948148,0.955224,0.951673


In [35]:
test_metrics, test_y_true, test_y_pred, test_pos_probs, test_confidences = evaluate_model(
    model, test_loader, device
)

metrics_table = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Precision (positive class)",
            "Recall (positive class)",
            "F1 (positive class)",
        ],
        "value": [
            test_metrics["accuracy"],
            test_metrics["precision_positive"],
            test_metrics["recall_positive"],
            test_metrics["f1_positive"],
        ],
    }
)

metrics_table["value"] = metrics_table["value"].map(lambda x: round(float(x), 4))
display(metrics_table)

print(
    f"Test accuracy = {test_metrics['accuracy']:.4f}\n"
    f"Test precision (positive) = {test_metrics['precision_positive']:.4f}\n"
    f"Test recall (positive) = {test_metrics['recall_positive']:.4f}\n"
    f"Test F1 (positive) = {test_metrics['f1_positive']:.4f}"
)

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,metric,value
0,Accuracy,0.9526
1,Precision (positive class),0.9502
2,Recall (positive class),0.9598
3,F1 (positive class),0.9550


Test accuracy = 0.9526
Test precision (positive) = 0.9502
Test recall (positive) = 0.9598
Test F1 (positive) = 0.9550


In [36]:
BEST_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(BEST_EXPORT_DIR)
tokenizer.save_pretrained(BEST_EXPORT_DIR)
print(f"Saved best model + tokenizer to: {BEST_EXPORT_DIR.resolve()}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model + tokenizer to: /content/bert_yelp_sentiment_best


In [37]:
def truncate_text(text: str, max_chars: int = 220) -> str:
    text = " ".join(str(text).split())
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."

def pick_diverse_examples(df: pd.DataFrame, group_col: str, max_total: int = 5, base_per_group: int = 2) -> pd.DataFrame:
    picked_parts = []
    picked_indices = []

    for group_value in sorted(df[group_col].dropna().unique().tolist()):
        group_df = df[df[group_col] == group_value].sort_values("prediction_confidence", ascending=False)
        take = group_df.head(base_per_group)
        if not take.empty:
            picked_parts.append(take)
            picked_indices.extend(take.index.tolist())

    combined = pd.concat(picked_parts, axis=0) if picked_parts else df.head(0).copy()

    if len(combined) < max_total:
        remaining = df.drop(index=picked_indices, errors="ignore").sort_values(
            "prediction_confidence", ascending=False
        )
        combined = pd.concat([combined, remaining.head(max_total - len(combined))], axis=0)

    combined = combined.sort_values("prediction_confidence", ascending=False).head(max_total).copy()
    return combined

analysis_df = test_eval_df.copy().reset_index(drop=True)
analysis_df["true_label_id"] = test_y_true
analysis_df["predicted_label_id"] = test_y_pred
analysis_df["true_label"] = analysis_df["true_label_id"].map(ID_TO_LABEL_NAME)
analysis_df["predicted_label"] = analysis_df["predicted_label_id"].map(ID_TO_LABEL_NAME)
analysis_df["positive_probability"] = test_pos_probs
analysis_df["prediction_confidence"] = test_confidences
analysis_df["correct"] = analysis_df["true_label_id"] == analysis_df["predicted_label_id"]
analysis_df["text_truncated"] = analysis_df["text"].map(truncate_text)
analysis_df["error_type"] = np.where(
    analysis_df["correct"],
    "correct",
    np.where(
        (analysis_df["true_label_id"] == 1) & (analysis_df["predicted_label_id"] == 0),
        "false_negative",
        "false_positive",
    ),
)

full_test_token_lengths = tokenizer(
    analysis_df["text"].tolist(),
    truncation=False,
    add_special_tokens=True,
)["input_ids"]
analysis_df["token_length_full"] = [len(ids) for ids in full_test_token_lengths]
analysis_df["was_truncated_at_max_length"] = analysis_df["token_length_full"] > MAX_LENGTH

correct_examples = pick_diverse_examples(
    analysis_df[analysis_df["correct"]],
    group_col="true_label",
    max_total=5,
    base_per_group=2,
)[
    ["text_truncated", "true_label", "predicted_label", "positive_probability", "prediction_confidence"]
].copy()

error_examples = pick_diverse_examples(
    analysis_df[~analysis_df["correct"]],
    group_col="error_type",
    max_total=5,
    base_per_group=2,
)[
    ["text_truncated", "true_label", "predicted_label", "positive_probability", "prediction_confidence"]
].copy()

for df_ in [correct_examples, error_examples]:
    for col in ["positive_probability", "prediction_confidence"]:
        if col in df_.columns:
            df_[col] = df_[col].map(lambda x: round(float(x), 4))

print("Correctly classified review examples:")
display(correct_examples)

print("Misclassified review examples:")
display(error_examples)

print(f"Total test errors: {(~analysis_df['correct']).sum()} out of {len(analysis_df)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (575 > 512). Running this sequence through the model will result in indexing errors


Correctly classified review examples:


,text_truncated,true_label,predicted_label,positive_probability,prediction_confidence
368,"To quote ESPN... \""c'mon man!\"" How about putting a decent product on the floor. And where is Michael Jordan? Does he attend games? The night we went they were playing the Celtics. BTW, we are BIG Celtics fans. It rem...",negative,negative,0.0010,0.9990
72,"Cons:\n* Very cramped space, aisles are so narrow that you have to carry your case sideways.\n* Only a few items were priced with signs hanging above them.\n* No shopping carts.\n\nPros:\n* Decent selection\n* Chilled...",negative,negative,0.0011,0.9989
111,"WARNING ,,,,, this place is for SUCKERS ONLY ..... funky smelling ,worse tasting seafood ... overpriced ..... raunchy,un-clean restrooms .... only reason anyone goes here is because this is a ''tourist trap '' type ne...",negative,negative,0.0011,0.9989
310,Really nice outdoor patio. Sunday drink special on whiskey was really good. Very reasonable price for uptown. Nice variations on Irish cuisine. Worth checking out!,positive,positive,0.9985,0.9985
313,This is a great stop for a meal if your going to a Bobcat's game or any event at Time Warner arena.\nThey have covered garage parking for just $5 and a menu that just won't quit. They've got something for everyone.\nT...,positive,positive,0.9985,0.9985


Misclassified review examples:


,text_truncated,true_label,predicted_label,positive_probability,prediction_confidence
264,Says they deliver on here... Wrong & wrong again & should not be checked! I like Applebee's & thought the delivery was something new for the Pittsburgh area... Don't know if this is something Yelp does or something so...,positive,negative,0.0015,0.9985
336,"I've gotten massages from Modern Salon several times, and loved them all there... until the last time, when I lost my penchant- after learning the masseuse I had used before, no longer worked there. His replacement wa...",positive,negative,0.0018,0.9982
270,"Get your wallet ready, the prices are crazy high!",negative,positive,0.9966,0.9966
238,Let me start by saying I was given a coupon for a free haircut and mani. You have to really let me down with free to get a two-star review.\n\nTHE HAIRCUT.\n Let's put it this way--after I went home and re-washed and ...,negative,positive,0.9961,0.9961
153,The reason for this update is the service. The hotel itself is still beautiful and the food is above average. All solid four stars. I was a bit disappointed initially with some of the pre-arrangements. In the past I w...,positive,negative,0.0043,0.9957


Total test errors: 18 out of 380


In [38]:
problem2_settings_df = pd.DataFrame(
    {
        "setting": [
            "pretrained_model_name",
            "pipeline_batch_size",
            "pipeline_max_length",
            "mapped_label_space",
        ],
        "value": [
            PIPELINE_MODEL_NAME,
            PIPELINE_BATCH_SIZE,
            MAX_LENGTH,
            "0 = negative, 1 = positive",
        ],
    }
)

display(problem2_settings_df)

PIPELINE_LABEL_TO_ID = {
    "NEGATIVE": 0,
    "POSITIVE": 1,
    "LABEL_0": 0,
    "LABEL_1": 1,
}

pipeline_mapping_df = pd.DataFrame(
    {
        "pipeline_output_label": list(PIPELINE_LABEL_TO_ID.keys()),
        "mapped_label_id": list(PIPELINE_LABEL_TO_ID.values()),
        "mapped_label_name": [ID_TO_LABEL_NAME[v] for v in PIPELINE_LABEL_TO_ID.values()],
    }
)

display(pipeline_mapping_df)

if device.type == "cuda" and "model" in globals():
    try:
        model.to("cpu")
        torch.cuda.empty_cache()
        gc.collect()
        print("Moved the fine-tuned BERT model to CPU to free GPU memory for Problem 2 inference.")
    except Exception as e:
        print(f"GPU cleanup warning: {e}")

sentiment_pipe = pipeline(
    task="sentiment-analysis",
    model=PIPELINE_MODEL_NAME,
    tokenizer=PIPELINE_MODEL_NAME,
    device=0 if device.type == "cuda" else -1,
)
print(f"Loaded pre-trained pipeline model: {PIPELINE_MODEL_NAME}")

,setting,value
0,pretrained_model_name,distilbert-base-uncased-finetuned-sst-2-english
1,pipeline_batch_size,32
2,pipeline_max_length,256
3,mapped_label_space,"0 = negative, 1 = positive"


,pipeline_output_label,mapped_label_id,mapped_label_name
0,NEGATIVE,0,negative
1,POSITIVE,1,positive
2,LABEL_0,0,negative
3,LABEL_1,1,positive


Moved the fine-tuned BERT model to CPU to free GPU memory for Problem 2 inference.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded pre-trained pipeline model: distilbert-base-uncased-finetuned-sst-2-english


In [39]:
from transformers import pipeline

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

print("Sentiment pipeline loaded.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentiment pipeline loaded.


In [40]:
def run_sentiment_pipeline_in_batches(pipe, texts, batch_size=32, max_length=256):
    outputs = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Problem 2 pipeline inference"):
        batch_texts = texts[start : start + batch_size]
        batch_outputs = pipe(batch_texts, truncation=True, max_length=max_length)
        if isinstance(batch_outputs, dict):
            batch_outputs = [batch_outputs]
        outputs.extend(batch_outputs)
    return outputs


def map_pipeline_label_to_id(label: str) -> int:
    key = str(label).upper()
    if key not in PIPELINE_LABEL_TO_ID:
        raise ValueError(f"Unexpected pipeline label: {label}")
    return PIPELINE_LABEL_TO_ID[key]


problem2_outputs = run_sentiment_pipeline_in_batches(
    sentiment_pipe,
    test_eval_df["text"].tolist(),
    batch_size=PIPELINE_BATCH_SIZE,
    max_length=MAX_LENGTH,
)

problem2_df = test_eval_df.copy()
problem2_df["pipeline_label"] = [out["label"] for out in problem2_outputs]
problem2_df["pipeline_score"] = [float(out["score"]) for out in problem2_outputs]
problem2_df["predicted_label_id"] = [
    map_pipeline_label_to_id(label) for label in problem2_df["pipeline_label"]
]
problem2_df["true_label"] = problem2_df["label"].map(ID_TO_LABEL_NAME)
problem2_df["predicted_label"] = problem2_df["predicted_label_id"].map(ID_TO_LABEL_NAME)
problem2_df["correct"] = problem2_df["label"] == problem2_df["predicted_label_id"]
problem2_df["text_truncated"] = problem2_df["text"].map(truncate_text)

problem2_metrics = compute_binary_metrics(
    problem2_df["label"].to_numpy(),
    problem2_df["predicted_label_id"].to_numpy(),
)

problem2_metrics_table = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Precision (positive class)",
            "Recall (positive class)",
            "F1 (positive class)",
        ],
        "value": [
            problem2_metrics["accuracy"],
            problem2_metrics["precision_positive"],
            problem2_metrics["recall_positive"],
            problem2_metrics["f1_positive"],
        ],
    }
)
problem2_metrics_table["value"] = problem2_metrics_table["value"].map(lambda x: round(float(x), 4))

display(problem2_metrics_table)

print(
    f"Problem 2 — pre-trained model accuracy = {problem2_metrics['accuracy']:.4f}\n"
    f"Problem 2 — pre-trained model precision (positive) = {problem2_metrics['precision_positive']:.4f}\n"
    f"Problem 2 — pre-trained model recall (positive) = {problem2_metrics['recall_positive']:.4f}\n"
    f"Problem 2 — pre-trained model F1 (positive) = {problem2_metrics['f1_positive']:.4f}"
)

problem2_preview_df = problem2_df[
    ["text_truncated", "true_label", "pipeline_label", "predicted_label", "pipeline_score", "correct"]
].head(8).copy()
problem2_preview_df["pipeline_score"] = problem2_preview_df["pipeline_score"].map(lambda x: round(float(x), 4))

print("Sample Problem 2 predictions:")
display(problem2_preview_df)

Problem 2 pipeline inference:   0%|          | 0/12 [00:00<?, ?it/s]

,metric,value
0,Accuracy,0.8895
1,Precision (positive class),0.9067
2,Recall (positive class),0.8794
3,F1 (positive class),0.8929


Problem 2 — pre-trained model accuracy = 0.8895
Problem 2 — pre-trained model precision (positive) = 0.9067
Problem 2 — pre-trained model recall (positive) = 0.8794
Problem 2 — pre-trained model F1 (positive) = 0.8929
Sample Problem 2 predictions:


,text_truncated,true_label,pipeline_label,predicted_label,pipeline_score,correct
0,"Contrary to other reviews, I have zero complaints about the service or the prices. I have been getting tire service here for the past 5 years now, and compared to my experience with places like Pep Boys, these guys ar...",positive,NEGATIVE,negative,0.9933,False
1,"Last summer I had an appointment to get new tires and had to wait a super long time. I also went in this week for them to fix a minor problem with a tire they put on. They \""fixed\"" it for free, and the very next morn...",negative,NEGATIVE,negative,0.9979,True
2,"Friendly staff, same starbucks fair you get anywhere else. Sometimes the lines can get long.",positive,NEGATIVE,negative,0.9915,False
3,"The food is good. Unfortunately the service is very hit or miss. The main issue seems to be with the kitchen, the waiters and waitresses are often very apologetic for the long waits and it's pretty obvious that some o...",negative,NEGATIVE,negative,0.9985,True
4,"Even when we didn't have a car Filene's Basement was worth the bus trip to the Waterfront. I always find something (usually I find 3-4 things and spend about $60) and better still, I am always still wearing the clothe...",positive,POSITIVE,positive,0.9990,True
5,"Picture Billy Joel's \""Piano Man\"" DOUBLED mixed with beer, a rowdy crowd, and comedy - Welcome to Sing Sing! A unique musical experience found in Homestead.\n\nIf you're looking to grab a bite to eat or a beer, come ...",positive,POSITIVE,positive,0.9984,True
6,Mediocre service. COLD food! Our food waited so long the lettuce & pickles wilted. Bland food. Crazy overpriced. Long waits in the arcade. 1 beer per hour maximum. Avoid at all costs. Fair manager.,negative,NEGATIVE,negative,0.9997,True
7,"Ok! Let me tell you about my bad experience first. I went to D&B last night for a post wedding party - which, side note, is a great idea!\n\nIt was around midnight and the bar wasn't really populated. There were three...",negative,NEGATIVE,negative,0.9940,True


In [42]:
comparison_raw_df = pd.DataFrame(
    [
        {
            "Model": "Fine-tuned BERT (bert-base-uncased)",
            "Accuracy": test_metrics["accuracy"],
            "Precision": test_metrics["precision_positive"],
            "Recall": test_metrics["recall_positive"],
            "F1": test_metrics["f1_positive"],
        },
        {
            "Model": "Pre-trained sentiment model (distilbert-base-uncased-finetuned-sst-2-english)",
            "Accuracy": problem2_metrics["accuracy"],
            "Precision": problem2_metrics["precision_positive"],
            "Recall": problem2_metrics["recall_positive"],
            "F1": problem2_metrics["f1_positive"],
        },
    ]
)

comparison_display_df = comparison_raw_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1"]:
    comparison_display_df[col] = comparison_display_df[col].map(lambda x: round(float(x), 4))

display(comparison_display_df)

,Model,Accuracy,Precision,Recall,F1
0,Fine-tuned BERT (bert-base-uncased),0.9526,0.9502,0.9598,0.9550
1,Pre-trained sentiment model (distilbert-base-uncased-finetuned-sst-2-english),0.8895,0.9067,0.8794,0.8929


'\nmetric_columns = ["Accuracy", "Precision", "Recall", "F1"]\nmodel_names = comparison_raw_df["Model"].tolist()\nwinner_tally = {name: 0 for name in model_names}\nmetric_summary_lines = []\n\nfor metric in metric_columns:\n    metric_frame = comparison_raw_df[["Model", metric]].sort_values(metric, ascending=False).reset_index(drop=True)\n    top_model = metric_frame.loc[0, "Model"]\n    top_value = float(metric_frame.loc[0, metric])\n    second_value = float(metric_frame.loc[1, metric])\n    margin = top_value - second_value\n\n    if abs(margin) < 1e-12:\n        metric_summary_lines.append(f"- **{metric}:** tie at {top_value:.4f}.")\n    else:\n        winner_tally[top_model] += 1\n        metric_summary_lines.append(f"- **{metric}:** {top_model} is higher by {margin:.4f}.")\n\nfine_tuned_name = model_names[0]\npretrained_name = model_names[1]\n\nif winner_tally[fine_tuned_name] > winner_tally[pretrained_name]:\n    overall_text = (\n        "Across these four metrics, the **fine-tu